# Synthetic Data Generation for Fine-Tuning

## Why Synthetic Data?

High-quality labeled data is the bottleneck for fine-tuning. Human annotation is slow and expensive (~$0.10-10/sample).
The modern approach: use large models (GPT-4, Claude, Mixtral) to generate synthetic training data.

**Three patterns:**

1. **LLM-as-annotator**: Give unlabeled data to a strong model, get labeled outputs
2. **Self-instruct**: Use a model to generate instructions + responses from seed examples
3. **Knowledge distillation**: Use a large model's outputs to train a smaller student model

**Key insight from Stanford Alpaca (2023):** 52,000 GPT-3.5 instruction-response pairs ($500 to generate)
produced a 7B model competitive with GPT-3. The data quality > quantity principle was proven.

In [ ]:
!pip install transformers datasets trl -q 2>&1 | tail -2

In [ ]:
# Pattern 1: Self-instruct — generate instructions from seeds
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch, json, re

# Use a local model as the teacher (avoids API costs in this demo)
# In production: use GPT-4-turbo or Claude-3.5-sonnet via API
model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
gen = pipeline("text-generation", model=model_id, torch_dtype=torch.float16, device_map="auto", max_new_tokens=150)

# Seed examples for LLM fine-tuning domain
seeds = [
    "Explain LoRA",
    "What is attention in transformers",
    "How does gradient checkpointing work",
]

print("Generating synthetic instructions from seeds...")
generated_instructions = []

for seed in seeds:
    prompt = f"""You are an AI assistant helping create a dataset for LLM fine-tuning education.
Based on this topic: '{seed}'

Generate 3 different instructional questions that a learner might ask. Format as:
1. [question]
2. [question]  
3. [question]

Questions:"""
    
    output = gen(prompt, do_sample=True, temperature=0.8, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    response = output[len(prompt):]
    questions = re.findall(r'\d+\.\s+(.+?)(?=\n|$)', response)
    generated_instructions.extend(questions[:3])
    print(f"  From '{seed}': generated {len(questions)} questions")

print(f"\nTotal generated: {len(generated_instructions)} instructions")
for i, q in enumerate(generated_instructions[:6]):
    print(f"  {i+1}. {q.strip()}")

In [ ]:
# Pattern 2: LLM-as-annotator — generate responses for instructions
print("Generating responses for instructions...")

synthetic_dataset = []
RESPONSE_PROMPT = """You are an expert in machine learning and LLM fine-tuning. 
Answer the following question accurately and concisely (2-4 sentences).

Question: {question}

Answer:"""

for instruction in generated_instructions[:6]:  # limit for speed
    prompt = RESPONSE_PROMPT.format(question=instruction.strip())
    output = gen(prompt, do_sample=True, temperature=0.3, pad_token_id=tokenizer.eos_token_id)[0]["generated_text"]
    response = output[len(prompt):].strip()
    
    # Basic quality filter: response should be 20-300 chars
    if 20 < len(response) < 300:
        synthetic_dataset.append({
            "instruction": instruction.strip(),
            "response": response,
            "source": "synthetic-self-instruct"
        })

print(f"Generated {len(synthetic_dataset)} high-quality pairs")
for sample in synthetic_dataset[:2]:
    print(f"\nInstruction: {sample['instruction']}")
    print(f"Response: {sample['response'][:150]}...")

In [ ]:
# Pattern 3: Quality filtering — key for synthetic data
from datasets import Dataset
import re

def quality_filter(sample):
    r = sample["response"]
    # Remove samples with common issues
    if len(r) < 30: return False          # too short
    if len(r) > 500: return False         # too long  
    if r.count("\n") > 5: return False   # too many line breaks (probably hallucination)
    if "I don't know" in r: return False  # model uncertainty
    if "I cannot" in r: return False      # refusal
    return True

filtered = [s for s in synthetic_dataset if quality_filter(s)]
print(f"After quality filtering: {len(synthetic_dataset)} → {len(filtered)} samples")
print(f"Rejection rate: {(1 - len(filtered)/max(len(synthetic_dataset),1))*100:.0f}%")

# Save as HuggingFace dataset
if filtered:
    ds = Dataset.from_list(filtered)
    print(f"\nFinal dataset stats:")
    print(f"  Size: {len(ds)} samples")
    print(f"  Avg instruction length: {sum(len(s['instruction'].split()) for s in filtered)/len(filtered):.0f} words")
    print(f"  Avg response length: {sum(len(s['response'].split()) for s in filtered)/len(filtered):.0f} words")
    
    # ds.push_to_hub("your-username/my-synthetic-dataset")  # uncomment to publish

## Production-Scale Synthetic Data with Distilabel

For serious data generation, use HuggingFace's **Distilabel** pipeline:

```python
from distilabel.pipeline import Pipeline
from distilabel.steps import LoadDataFromHub
from distilabel.steps.tasks import TextGeneration

# Generate 10K instruction-response pairs using a teacher model
with Pipeline(name="synthetic-llm-data") as pipeline:
    loader = LoadDataFromHub(
        repo_id="HuggingFaceH4/instruction-mix",
        split="train",
        batch_size=32,
    )
    generator = TextGeneration(
        llm=OpenAILLM(model="gpt-4o-mini"),  # teacher model
        system_prompt="You are an expert in LLMs. Answer accurately and concisely.",
        num_generations=1,
    )
    loader >> generator

distiset = pipeline.run(parameters={"TextGeneration": {"batch_size": 32}})
distiset.push_to_hub("your-username/synthetic-llm-dataset")
```

**Cost estimate (10K samples):**
- GPT-4o-mini: ~$2-5
- GPT-4-turbo: ~$40-80
- Llama-70B API: ~$1-3
- Local Qwen-72B: free (but slow)

**Quality tip:** Self-consistency filtering — generate 3 responses and keep only ones where
2+ responses agree. Boosts quality from ~85% to ~95% accuracy at 3× compute cost.